# Train balance bot using PPO with commands

Run each cell by pressing `shift + enter`.

We repeat our training but with added commands. We spend the first couple of phases getting the bot to balance (without any velocity or turning commands) before adding in the commands. Once the bot handles commands in an ideal environment, we add domain randomization (DR) back in.

Important takeaway: make sure the bot is behaving as intended at the end of each phase before moving on!

In [1]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

# Import custom environment
from balance_bot_env_cmd import BalanceBotEnv, DomainRandomConfig

# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import PPO training module and exporter
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx
from rl.onnx_actor_to_c import export_onnx_actor_to_c

In [2]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 8               # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 500_000    # Number of simulation steps to perform per environment

## Configure PPO and environment

In [3]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 48,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 48,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [4]:
def make_balance_bot_env(render, debug, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path = MJCF_PATH,
        render_mode = "human" if render else None,
        debug = debug,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), debug=(i==0), kw=kwargs: 
                make_balance_bot_env(render, debug=debug, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [5]:
def load_agent(envs, model_path):
    """
    For debugging only! Use this to load a previously trained model to skip previous phases. Note 
    that you will still need to run the cells in each prior phase that update the experiment name 
    and environment.
    """
    from rl.ppo_trainer import Agent, TrainResult

    # Make sure model_path is a Path
    model_path = Path(model_path)
    
    # Load agent from previous run
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent = Agent(envs, ppo_config).to(device)
    agent.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    agent.eval()
    print(f"Loaded model from {model_path}")
    
    # Wrap agent in a dummy result
    return TrainResult(
        agent = agent,
        checkpoint_dir = model_path.parent,
        best_model_path = model_path,
        final_model_path = None,
        best_mean_return = 0,
    )

## Phase 1: Balance only

We'll start with the easy task of only penalizing if the robot tilts (pitches) forward.

In [6]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-01"

# Create an environment that only rewards staying upright
envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    vel_reward_coef=0.0,
    yaw_reward_coef=0.0,
)

In [7]:
# Choo choo train
result = train(ppo_config, envs=envs)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-01__42__1780495139
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-01
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.049 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=-0.004 rad/s
  vel_tracking_reward=0.0000 (perfect=1.0)
  yaw_tracking_reward=0.0000 (perfect=1.0)
  pitch=0.092 rad, pitch_penalty=0.0042
  pitch_rate=0.786 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.159 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=1.602 rad/s
  vel_tracking_reward=0.0000 (perfect=1.0)
  yaw_tracking_reward=0.0000 (perfect=1.0)
  pitch=0.080 rad, pitch_penalty=0.0032
  pitch_rate=1.158 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.068 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=-1.446 rad/s
  vel_

In [8]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.019 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=2.171 rad/s
  vel_tracking_reward=0.0000 (perfect=1.0)
  yaw_tracking_reward=0.0000 (perfect=1.0)
  pitch=-0.021 rad, pitch_penalty=0.0002
  pitch_rate=-0.086 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.015 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=2.128 rad/s
  vel_tracking_reward=0.0000 (perfect=1.0)
  yaw_tracking_reward=0.0000 (perfect=1.0)
  pitch=0.002 rad, pitch_penalty=0.0000
  pitch_rate=-0.740 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.027 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=1.831 rad/s
  vel_tracking_reward=0.0000 (perfect=1.0)
  yaw_tracking_reward=0.0000 (perfect=1.0)
  pitch=-0.023 rad, pitch_penalty=0.0003
  pitch_rate=0.

## Phase 2: Penalize position and rotation

In [9]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-02"

# Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.position_penalty_coef = 0.01
    env.pitch_rate_penalty_coef = 0.02
    env.vel_reward_coef = 1.0
    env.yaw_reward_coef = 1.0
    env.sigma_vel = 0.1
    env.sigma_yaw = 0.5
    env.vel_max = 0.5

In [10]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-02__42__1780496549
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-02
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.026 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=1.745 rad/s
  vel_tracking_reward=0.9933 (perfect=1.0)
  yaw_tracking_reward=0.0023 (perfect=1.0)
  pitch=0.002 rad, pitch_penalty=0.0000
  pitch_rate=-0.228 rad/s, pitch_rate_penalty=0.0010
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.020 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=2.160 rad/s
  vel_tracking_reward=0.9960 (perfect=1.0)
  yaw_tracking_reward=0.0001 (perfect=1.0)
  pitch=0.007 rad, pitch_penalty=0.0000
  pitch_rate=0.187 rad/s, pitch_rate_penalty=0.0007
  action_smoothness_penalty=0.0000
--- step 3000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.053 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=1.031 rad/s
  vel

In [11]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.005 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.011 rad/s
  vel_tracking_reward=0.9998 (perfect=1.0)
  yaw_tracking_reward=0.9998 (perfect=1.0)
  pitch=0.028 rad, pitch_penalty=0.0004
  pitch_rate=0.217 rad/s, pitch_rate_penalty=0.0009
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.032 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.007 rad/s
  vel_tracking_reward=0.9896 (perfect=1.0)
  yaw_tracking_reward=0.9999 (perfect=1.0)
  pitch=0.015 rad, pitch_penalty=0.0001
  pitch_rate=-0.110 rad/s, pitch_rate_penalty=0.0002
  action_smoothness_penalty=0.0000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.015 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.062 rad/s
  vel_tracking_reward=0.9978 (perfect=1.0)
  yaw_tracking_reward=0.9923 (perfect=1.0)
  pitch=0.003 rad, pitch_penalty=0.0000
  pitch_rate=0.135 ra

## Phase 3: Small velocity command, penalize pitch rate and rotation (yaw)

In [12]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-03"

# Update domain randomization config
dr = DomainRandomConfig(
    cmd_vel_range=(-0.5, 0.5),  # Slow max forward/back commands
    cmd_yaw_range=(0.0, 0.0),   # No yaw commands yet
    cmd_zero_prob=0.5,          # Ensure bot stays still 50% of the time
)

# Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr
    env.position_penalty_coef = 0.01
    env.pitch_rate_penalty_coef = 0.02
    env.vel_reward_coef = 1.0
    env.yaw_reward_coef = 1.0
    env.sigma_vel = 0.1
    env.sigma_yaw = 0.5
    env.vel_max = 0.5

# Update the PPO config to help prevent getting stuck in a local maximum
ppo_config.ent_coef = 0.01      # Encourage the agent to explore more  
ppo_config.value_clip = 10.0    # Allow for more aggressive critic updates

In [13]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-01__42__1779580717/best_model.cleanrl_model")

In [14]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-03__42__1780498424
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-03
New episode: cmd_vel=-0.061, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=-0.061, vel_target=-0.031 m/s, vel_actual=0.037 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=-0.046 rad/s
  vel_tracking_reward=0.9549 (perfect=1.0)
  yaw_tracking_reward=0.9958 (perfect=1.0)
  pitch=0.033 rad, pitch_penalty=0.0005
  pitch_rate=-0.049 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=-0.061, vel_target=-0.031 m/s, vel_actual=0.038 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.060 rad/s
  vel_tracking_reward=0.9536 (perfect=1.0)
  yaw_tracking_reward=0.9927 (perfect=1.0)
  pitch=0.023 rad, pitch_penalty=0.0003
  pitch_rate=0.127 rad/s, pitch_rate_penalty=0.0003
  action_smoothness_penalty=0.0000
--- step 3000 ---
  cmd_vel=-0.061, vel_target=-0.031 m/s, vel_actual=0.028 m/s
  cmd_yaw=0.000, yaw_t

In [15]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=-0.184, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=-0.184, vel_target=-0.092 m/s, vel_actual=-0.073 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.041 rad/s
  vel_tracking_reward=0.9966 (perfect=1.0)
  yaw_tracking_reward=0.9967 (perfect=1.0)
  pitch=-0.035 rad, pitch_penalty=0.0006
  pitch_rate=0.316 rad/s, pitch_rate_penalty=0.0020
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=-0.184, vel_target=-0.092 m/s, vel_actual=-0.072 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.033 rad/s
  vel_tracking_reward=0.9959 (perfect=1.0)
  yaw_tracking_reward=0.9978 (perfect=1.0)
  pitch=-0.032 rad, pitch_penalty=0.0005
  pitch_rate=-0.016 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
New episode: cmd_vel=0.000, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.021 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.065 rad/s
  vel_tracking_reward=0.9955 (perfect=1.0)
  yaw_track

## Phase 4: Introduce small yaw commands and full velocity commands

In [16]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-04"

# Update domain randomization config
dr.cmd_vel_range = (-1.0, 1.0)
dr.cmd_yaw_range = (-0.5, 0.5)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr
    env.sigma_yaw = 0.1         # Tighten to punish deviating from yaw command
    env.yaw_max = 1.5           # Slow max turn (rad/s)

In [17]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-04__42__1780499761
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-04
New episode: cmd_vel=-0.122, cmd_yaw=0.359
--- step 1000 ---
  cmd_vel=-0.122, vel_target=-0.061 m/s, vel_actual=-0.040 m/s
  cmd_yaw=0.359, yaw_target=0.538 rad/s, yaw_actual=-0.006 rad/s
  vel_tracking_reward=0.9956 (perfect=1.0)
  yaw_tracking_reward=0.0521 (perfect=1.0)
  pitch=-0.024 rad, pitch_penalty=0.0003
  pitch_rate=-0.041 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=-0.122, vel_target=-0.061 m/s, vel_actual=-0.033 m/s
  cmd_yaw=0.359, yaw_target=0.538 rad/s, yaw_actual=0.047 rad/s
  vel_tracking_reward=0.9919 (perfect=1.0)
  yaw_tracking_reward=0.0903 (perfect=1.0)
  pitch=-0.021 rad, pitch_penalty=0.0002
  pitch_rate=0.146 rad/s, pitch_rate_penalty=0.0004
  action_smoothness_penalty=0.0000
--- step 3000 ---
  cmd_vel=-0.122, vel_target=-0.061 m/s, vel_actual=-0.031 m/s
  cmd_yaw=0.359, 

In [18]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.000, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.003 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.045 rad/s
  vel_tracking_reward=0.9999 (perfect=1.0)
  yaw_tracking_reward=0.9801 (perfect=1.0)
  pitch=-0.003 rad, pitch_penalty=0.0000
  pitch_rate=0.285 rad/s, pitch_rate_penalty=0.0016
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.000 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.033 rad/s
  vel_tracking_reward=1.0000 (perfect=1.0)
  yaw_tracking_reward=0.9894 (perfect=1.0)
  pitch=-0.004 rad, pitch_penalty=0.0000
  pitch_rate=-0.126 rad/s, pitch_rate_penalty=0.0003
  action_smoothness_penalty=0.0000
New episode: cmd_vel=0.497, cmd_yaw=0.391
--- step 1000 ---
  cmd_vel=0.497, vel_target=0.249 m/s, vel_actual=0.047 m/s
  cmd_yaw=0.391, yaw_target=0.586 rad/s, yaw_actual=0.369 rad/s
  vel_tracking_reward=0.6663 (perfect=1.0)
  yaw_tracking_r

## Phase 5: Mid-episode command sampling

Keep same vel/yaw command range. Any larger, and it seems the bot will tip over. Add small chance of generating a new set of commands every few episodes.

In [19]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-05"

# Update domain randomization config
dr.cmd_yaw_range = (-1.0, 1.0)
dr.cmd_resample_prob = 0.005    # 0.5% chance of new command every step

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr

In [20]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-04__42__1779748283/best_model.cleanrl_model")

In [21]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-05__42__1780501205
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-05
New episode: cmd_vel=-0.122, cmd_yaw=0.717
New command: cmd_vel=0.355, cmd_yaw=-0.941
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.021, cmd_yaw=-0.334
New command: cmd_vel=0.238, cmd_yaw=-0.675
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.438, cmd_yaw=-0.876
--- step 1000 ---
  cmd_vel=-0.438, vel_target=-0.219 m/s, vel_actual=0.004 m/s
  cmd_yaw=-0.876, yaw_target=-1.314 rad/s, yaw_actual=-0.573 rad/s
  vel_tracking_reward=0.6072 (perfect=1.0)
  yaw_tracking_reward=0.0041 (perfect=1.0)
  pitch=-0.024 rad, pitch_penalty=0.0003
  pitch_rate=-0.302 rad/s, pitch_rate_penalty=0.0018
  action_smoothness_penalty=0.0000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.988, cmd_yaw=-0.923
New command: cmd_vel=0.000, c

In [22]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.006 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.031 rad/s
  vel_tracking_reward=0.9996 (perfect=1.0)
  yaw_tracking_reward=0.9904 (perfect=1.0)
  pitch=-0.013 rad, pitch_penalty=0.0001
  pitch_rate=0.405 rad/s, pitch_rate_penalty=0.0033
  action_smoothness_penalty=0.0000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.419, cmd_yaw=0.449
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.843, cmd_yaw=-0.719
New command: cmd_vel=0.026, cmd_yaw=-0.575
--- step 2000 ---
  cmd_vel=0.026, vel_target=0.013 m/s, vel_actual=-0.101 m/s
  cmd_yaw=-0.575, yaw_target=-0.863 rad/s, yaw_actual=-0.710 rad/s
  vel_tracking_reward=0.8792 (perfect=1.0)
  yaw_tracking_reward=0.7911 (perfect=1.0)
  pitch=0.024 rad, pitch_penalty=0.0003
  pitch_rate=0.141 rad/s, pitch_

## Phase 6: Observation noise and action delay

In [23]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-06"

# Update domain randomization config
dr.pitch_noise_std_dev=0.01         # Inject some noise into pitch observation
dr.pitch_rate_noise_std_dev=0.01    # Inject some noise into pitch rate observation
dr.wheel_vel_noise_std_dev = 0.1    # Inject some noise into wheel velocity observation
dr.action_delay_steps=1             # 1 step (5ms) delay
dr.action_delay_random=True         # Vary 0-1 steps each episode

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr
    env.smoothness_penalty_coef = 0.2  # Penalize rapid motion changes (jitter)

In [24]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-06__42__1780502709
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-06
New episode: cmd_vel=-0.122, cmd_yaw=0.717
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.898, cmd_yaw=0.513
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.436, cmd_yaw=0.134
New command: cmd_vel=-0.960, cmd_yaw=-0.163
New command: cmd_vel=-0.390, cmd_yaw=-0.639
New command: cmd_vel=-0.499, cmd_yaw=0.492
--- step 1000 ---
  cmd_vel=-0.499, vel_target=-0.250 m/s, vel_actual=-0.030 m/s
  cmd_yaw=0.492, yaw_target=0.738 rad/s, yaw_actual=0.770 rad/s
  vel_tracking_reward=0.6172 (perfect=1.0)
  yaw_tracking_reward=0.9899 (perfect=1.0)
  pitch=0.049 rad, pitch_penalty=0.0006
  pitch_rate=-0.103 rad/s, pitch_rate_penalty=0.0002
  action_smoothness_penalty=0.0062
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.853, cmd_y

In [25]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=-0.607, cmd_yaw=-0.086
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.576, cmd_yaw=-0.187
New command: cmd_vel=0.186, cmd_yaw=0.536
New command: cmd_vel=0.000, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.009 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=-0.023 rad/s
  vel_tracking_reward=0.9992 (perfect=1.0)
  yaw_tracking_reward=0.9945 (perfect=1.0)
  pitch=-0.020 rad, pitch_penalty=0.0002
  pitch_rate=0.305 rad/s, pitch_rate_penalty=0.0019
  action_smoothness_penalty=0.0107
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.230, cmd_yaw=-0.550
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.762, cmd_yaw=-0.755
New command: cmd_vel=-0.463, cmd_yaw=-0.619
--- step 2000 ---
  cmd_vel=-0.463, vel_target

## Phase 7: Add motor noise and random pushes

In [26]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-07"

# Update domain randomization config
dr.motor_noise_scale = 0.02 # +/-2% motor noise
dr.push_prob = 0.005        # 0.5% chance of push per step
dr.push_force_max_n = 0.3   # Gentle pushes

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr

In [27]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-07__42__1780504285
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-07
New episode: cmd_vel=-0.122, cmd_yaw=0.717
New command: cmd_vel=0.988, cmd_yaw=-0.923
New command: cmd_vel=0.220, cmd_yaw=-0.742
New command: cmd_vel=0.436, cmd_yaw=0.134
New command: cmd_vel=-0.960, cmd_yaw=-0.163
New command: cmd_vel=-0.499, cmd_yaw=0.492
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.853, cmd_yaw=0.953
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.126 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.018 rad/s
  vel_tracking_reward=0.8530 (perfect=1.0)
  yaw_tracking_reward=0.9967 (perfect=1.0)
  pitch=0.044 rad, pitch_penalty=0.0013
  pitch_rate=0.329 rad/s, pitch_rate_penalty=0.0022
  action_smoothness_penalty=0.0050
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=

In [28]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=-0.902, cmd_yaw=0.958
New command: cmd_vel=-0.580, cmd_yaw=-0.466
--- step 1000 ---
  cmd_vel=-0.580, vel_target=-0.290 m/s, vel_actual=-0.133 m/s
  cmd_yaw=-0.466, yaw_target=-0.699 rad/s, yaw_actual=-0.727 rad/s
  vel_tracking_reward=0.7824 (perfect=1.0)
  yaw_tracking_reward=0.9921 (perfect=1.0)
  pitch=-0.024 rad, pitch_penalty=0.0003
  pitch_rate=0.154 rad/s, pitch_rate_penalty=0.0005
  action_smoothness_penalty=0.0046
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.238, cmd_yaw=0.608
--- step 2000 ---
  cmd_vel=0.238, vel_target=0.119 m/s, vel_actual=0.008 m/s
  cmd_yaw=0.608, yaw_target=0.912 rad/s, yaw_actual=1.081 rad/s
  vel_tracking_reward=0.8853 (perfect=1.0)
  yaw_tracking_reward=0.7519 (perfect=1.0)
  pitch=0.023 rad, pitch_penalty=0.0000
  pitch_rate=0.196 rad/s, pitch_rate_penalty=0.0008
  action_smoothness_penalty=0.0009
New episode: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.295, cmd_yaw=-0.409
New command: cmd_vel=0.000,

## Phase 8: Add mass and friction randomization

In [29]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-08"

# Update domain randomization config
dr.mass_scale_range= (0.8, 1.2)       # +/-20% mass variation
dr.friction_scale_range = (0.5, 1.5)  # 50-150% friction variation

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr

In [30]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-08__42__1780506013
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-08
New episode: cmd_vel=0.395, cmd_yaw=-0.812
New command: cmd_vel=0.355, cmd_yaw=-0.941
New command: cmd_vel=0.988, cmd_yaw=-0.923
New episode: cmd_vel=0.586, cmd_yaw=0.239
New command: cmd_vel=-0.960, cmd_yaw=-0.163
New command: cmd_vel=-0.499, cmd_yaw=0.492
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.853, cmd_yaw=0.953
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.002 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.110 rad/s
  vel_tracking_reward=0.9999 (perfect=1.0)
  yaw_tracking_reward=0.8851 (perfect=1.0)
  pitch=-0.001 rad, pitch_penalty=0.0000
  pitch_rate=-0.195 rad/s, pitch_rate_penalty=0.0008
  action_smoothness_penalty=0.0046
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw

In [31]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.703, cmd_yaw=0.456
New command: cmd_vel=-0.580, cmd_yaw=-0.466
--- step 1000 ---
  cmd_vel=-0.580, vel_target=-0.290 m/s, vel_actual=-0.148 m/s
  cmd_yaw=-0.466, yaw_target=-0.699 rad/s, yaw_actual=-0.815 rad/s
  vel_tracking_reward=0.8182 (perfect=1.0)
  yaw_tracking_reward=0.8740 (perfect=1.0)
  pitch=-0.050 rad, pitch_penalty=0.0016
  pitch_rate=0.116 rad/s, pitch_rate_penalty=0.0003
  action_smoothness_penalty=0.0068
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.238, cmd_yaw=0.608
--- step 2000 ---
  cmd_vel=0.238, vel_target=0.119 m/s, vel_actual=0.023 m/s
  cmd_yaw=0.608, yaw_target=0.912 rad/s, yaw_actual=1.044 rad/s
  vel_tracking_reward=0.9120 (perfect=1.0)
  yaw_tracking_reward=0.8394 (perfect=1.0)
  pitch=0.037 rad, pitch_penalty=0.0015
  pitch_rate=0.091 rad/s, pitch_rate_penalty=0.0002
  action_smoothness_penalty=0.0007
New episode: cmd_vel=-0.701, cmd_yaw=-0.178
New command: cmd_vel=0.295, cmd_yaw=-0.409
New command: cmd_vel=0.000

## Phase 9: Add motor gain randomization

In [32]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-09"

# Update domain randomization config
dr.motor_gain_range = (0.6, 1.0)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr

In [33]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-08__42__1779826517/best_model.cleanrl_model")

In [34]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-09__42__1780507773
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-09
New episode: cmd_vel=-0.812, cmd_yaw=0.951
New command: cmd_vel=0.355, cmd_yaw=-0.941
New command: cmd_vel=0.988, cmd_yaw=-0.923
New episode: cmd_vel=0.944, cmd_yaw=-0.505
New command: cmd_vel=0.220, cmd_yaw=-0.742
New episode: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.960, cmd_yaw=-0.163
New command: cmd_vel=-0.499, cmd_yaw=0.492
New episode: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.379, cmd_yaw=0.124
New command: cmd_vel=0.000, cmd_yaw=0.000
New episode: cmd_vel=-0.214, cmd_yaw=-0.764
New command: cmd_vel=-0.408, cmd_yaw=0.115
New episode: cmd_vel=0.585, cmd_yaw=-0.469
New episode: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.655, cmd_yaw=-0.883
New episode: cmd_vel=-0.753, cmd_yaw=0.546
New episode: cmd_vel=0.201, cmd_yaw=0.760
New com

In [35]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.580, cmd_yaw=-0.466
--- step 1000 ---
  cmd_vel=-0.580, vel_target=-0.290 m/s, vel_actual=-0.175 m/s
  cmd_yaw=-0.466, yaw_target=-0.699 rad/s, yaw_actual=-0.587 rad/s
  vel_tracking_reward=0.8769 (perfect=1.0)
  yaw_tracking_reward=0.8840 (perfect=1.0)
  pitch=-0.045 rad, pitch_penalty=0.0012
  pitch_rate=0.099 rad/s, pitch_rate_penalty=0.0002
  action_smoothness_penalty=0.0188
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.238, cmd_yaw=0.608
--- step 2000 ---
  cmd_vel=0.238, vel_target=0.119 m/s, vel_actual=-0.051 m/s
  cmd_yaw=0.608, yaw_target=0.912 rad/s, yaw_actual=0.855 rad/s
  vel_tracking_reward=0.7497 (perfect=1.0)
  yaw_tracking_reward=0.9683 (perfect=1.0)
  pitch=0.031 rad, pitch_penalty=0.0000
  pitch_rate=-0.006 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0132
New command: cmd_vel=0.295,

## Phase 10: Random axle torques to simulate ridges

In [36]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-10"

# Update domain randomization config
dr.ridge_prob=0.05              # 5% possibility of tire "ridge"
dr.ridge_torque_max_nm=0.005    # Add a slight torque to the joints (simulate tire ridges)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr
    env.smoothness_penalty_coef = 0.5  # Penalize rapid motion changes (jitter)

In [37]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-09__42__1779828587/best_model.cleanrl_model")

In [38]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-10__42__1780509711
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-10
New episode: cmd_vel=-0.812, cmd_yaw=0.951
New command: cmd_vel=0.355, cmd_yaw=-0.941
New command: cmd_vel=-0.438, cmd_yaw=-0.876
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.898, cmd_yaw=0.513
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.499, cmd_yaw=0.492
New command: cmd_vel=0.728, cmd_yaw=0.027
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.408, cmd_yaw=0.115
--- step 1000 ---
  cmd_vel=-0.408, vel_target=-0.204 m/s, vel_actual=-0.105 m/s
  cmd_yaw=0.115, yaw_target=0.172 rad/s, yaw_actual=0.240 rad/s
  vel_tracking_reward=0.9060 (perfect=1.0)
  yaw_tracking_reward=0.9550 (perfect=1.0)
  pitch=-0.052 rad, pitch_penalty=0.0017
  pitch_rate=0.338 rad/s, pitch_rate_penalty=0.0023
  action_smoothness_penalty=0.0129
New command: cmd_vel=-0.246, cmd_yaw=0.585
New command: cmd_vel=0.000, cmd_y

In [39]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3,
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.608, cmd_yaw=-0.055
New command: cmd_vel=0.106, cmd_yaw=-0.767
New command: cmd_vel=0.387, cmd_yaw=-0.907
--- step 1000 ---
  cmd_vel=0.387, vel_target=0.193 m/s, vel_actual=0.051 m/s
  cmd_yaw=-0.907, yaw_target=-1.360 rad/s, yaw_actual=-1.442 rad/s
  vel_tracking_reward=0.8164 (perfect=1.0)
  yaw_tracking_reward=0.9344 (perfect=1.0)
  pitch=-0.010 rad, pitch_penalty=0.0004
  pitch_rate=0.307 rad/s, pitch_rate_penalty=0.0019
  action_smoothness_penalty=0.0300
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.958, cmd_yaw=0.316
New command: cmd_vel=-0.780, cmd_yaw=-0.035
New command: cmd_vel=0.469, cmd_yaw=0.812
New command: cmd_vel=-0.871, cmd_yaw=0.849
New command: cmd_vel=-0.510, cmd_yaw=0.153
New command: cmd_vel=0.000, cmd_yaw=0.000
--- step 2000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.013 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actu

## Clean up and save model

At this point, we are done with training. We want to delete the environments and save the best actor from our final training phase. We'll export this actor network as an ONNX file that can be used on a variety of hardware platforms.

In [40]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

Closing env 0
Closing env 1
Closing env 2
Closing env 3
Closing env 4
Closing env 5
Closing env 6
Closing env 7


In [41]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

/workspace/software/rl/ppo_trainer.py:381: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0603 18:40:07.652000 78796 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0603 18:40:07.654000 78796 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0603 18:40:07.655000 78796 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Actor exported to ONNX: runs/BalanceBot-v0__balance-bot-phase-10__42__1780509711/actor.onnx


/opt/pyenv/versions/3.12.13/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [42]:
# Export actor to .h file
export_onnx_actor_to_c(
    onnx_path   = result.checkpoint_dir / "actor.onnx",
    output_path = result.checkpoint_dir / "actor.h"
)

Weights found:
  0.weight: shape=(48, 6), dtype=float32
  0.bias: shape=(48,), dtype=float32
  2.weight: shape=(48, 48), dtype=float32
  2.bias: shape=(48,), dtype=float32
  4.weight: shape=(2, 48), dtype=float32
  4.bias: shape=(2,), dtype=float32
C header written to runs/BalanceBot-v0__balance-bot-phase-10__42__1780509711/actor.h
  Layers:  3
  Obs:     6
  Actions: 2


PosixPath('runs/BalanceBot-v0__balance-bot-phase-10__42__1780509711/actor.h')